# Análisis Exploratorio de Datos - Hubway Bike Sharing

## Autor
Guerra Chura Joan Leonardo

## Objetivo
Realizar un análisis exploratorio del sistema Hubway de bicicletas compartidas de Boston (2011-2013), aplicando técnicas de Data Wrangling, detección de problemas de calidad, visualización y análisis espacio-temporal.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

from scipy import stats

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)


# Paso 0: Metadata

El dataset está compuesto por:

- `hubway_stations.csv`: información geográfica de estaciones.
- `hubway_trips.csv`: registro individual de viajes.

Unidad de análisis:

- Una fila de trips representa un viaje.
- Una fila de stations representa una estación.

La relación se realiza mediante las estaciones de inicio y fin del viaje.

In [ ]:
stations = pd.read_csv("../data/hubway_stations.csv")
trips = pd.read_csv("../data/hubway_trips.csv", low_memory=False)

print("Stations:", stations.shape)
print("Trips:", trips.shape)


# Paso 1: Análisis del comportamiento de datos

Se revisan dimensiones, tipos, valores faltantes, duplicados y granularidad temporal y espacial.

In [ ]:
trips.info()

print("Duplicados trips:", trips.duplicated().sum())
print("Duplicados stations:", stations.duplicated().sum())


In [ ]:
trips['start_dt'] = pd.to_datetime(trips['start_date'])
trips['end_dt'] = pd.to_datetime(trips['end_date'])

missing = trips.isna().sum().sort_values(ascending=False)
missing


In [ ]:
plt.figure(figsize=(8,4))
(missing[missing>0]/len(trips)*100).plot(kind='bar')
plt.title("Porcentaje de valores faltantes")
plt.ylabel("%")
plt.xticks(rotation=45)
plt.show()


## Hallazgo de calidad

Los valores faltantes no deben asumirse siempre como errores. Algunas variables demográficas presentan ausencia asociada al tipo de usuario, por lo que pueden representar una característica del proceso de captura.

In [ ]:
pd.crosstab(trips['subsc_type'], trips['gender'].isna(), normalize='index')


# Paso 2: Análisis de Outliers

La variable principal será duration.

Se diferencian:

- Errores de datos: valores negativos o imposibles.
- Eventos operativos: viajes extremadamente largos.

In [ ]:
q1 = trips.duration.quantile(.25)
q3 = trips.duration.quantile(.75)
iqr = q3-q1

limite = q3 + 1.5*iqr

print("Q1:",q1)
print("Q3:",q3)
print("Límite superior:",limite)
print("Negativos:",(trips.duration<0).sum())
print(">24 horas:",(trips.duration>86400).sum())


In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(x=trips[trips.duration<86400]['duration'])
plt.title("Boxplot duración de viajes")
plt.show()


# Paso 3: Visualización

Se analizan patrones de usuarios, duración, tiempo y espacio.

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=trips, x='subsc_type')
plt.title("Viajes por tipo de usuario")
plt.show()


In [ ]:
trips['hour']=trips.start_dt.dt.hour

plt.figure(figsize=(8,4))
trips.hour.value_counts().sort_index().plot(kind='bar')
plt.title("Demanda por hora del día")
plt.xlabel("Hora")
plt.ylabel("Viajes")
plt.show()


In [ ]:
pivot = trips.assign(
    weekday=trips.start_dt.dt.day_name()
).pivot_table(
    index='weekday',
    columns='hour',
    values='hubway_id',
    aggfunc='count'
)

plt.figure(figsize=(12,5))
sns.heatmap(pivot)
plt.title("Heatmap hora vs día de semana")
plt.show()


In [ ]:
top = trips.strt_statn.value_counts().head(10)

plt.figure(figsize=(8,4))
top.plot(kind='bar')
plt.title("Top estaciones por salidas")
plt.xlabel("Estación")
plt.ylabel("Viajes")
plt.show()


# Paso 4: Problema potencial encontrado

## Influencia del clima y ubicación geográfica en la demanda

Pregunta:

¿Cómo afectan las condiciones meteorológicas y la localización de estaciones al número de viajes diarios?

Se incorpora una fuente externa climática para enriquecer el análisis.

In [ ]:
# Preparación de viajes diarios
trips['fecha']=trips.start_dt.dt.date

daily = trips.groupby('fecha').size().reset_index(name='n_viajes')
daily.head()


In [ ]:
# Consulta Open-Meteo
params = {
'latitude':42.355,
'longitude':-71.065,
'start_date':'2011-07-28',
'end_date':'2013-11-30',
'daily':'temperature_2m_max,precipitation_sum,snowfall_sum',
'timezone':'America/New_York'
}

r=requests.get("https://archive-api.open-meteo.com/v1/archive",params=params)

clima=pd.DataFrame(r.json()['daily'])
clima['time']=pd.to_datetime(clima['time']).dt.date

clima.head()


In [ ]:
clima_viajes=daily.merge(clima,left_on='fecha',right_on='time')

sns.scatterplot(
    data=clima_viajes,
    x='temperature_2m_max',
    y='n_viajes'
)
plt.title("Temperatura máxima vs viajes diarios")
plt.show()

sns.scatterplot(
    data=clima_viajes,
    x='precipitation_sum',
    y='n_viajes'
)
plt.title("Precipitación vs viajes diarios")
plt.show()


In [ ]:
print("Correlación temperatura:",
      clima_viajes.n_viajes.corr(clima_viajes.temperature_2m_max))

print("Correlación lluvia:",
      clima_viajes.n_viajes.corr(clima_viajes.precipitation_sum))


# Conclusión

El análisis permitió identificar problemas de calidad de datos, valores extremos, patrones temporales y espaciales.

El enriquecimiento con datos climáticos permite evaluar factores externos que afectan la demanda del sistema Hubway y genera una base para futuros modelos predictivos.